In [2]:
# 03 Weather Data Integration
#This notebook validates weather data availability for the project and prepares hourly Auckland weather variables for integration with GTFS transport delay data. This belongs to Layer 1 and Layer 2 of the system architecture only.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

WindowsPath('d:/Yoobee/Term 3/Capstone/ai-dss-auckland-public-transport')

Meteostat was installed in the capstone-gtfs environment using pip.

In [4]:
from meteostat import Point, Hourly
from datetime import datetime

print("Meteostat import successful")

Meteostat import successful


In [5]:
import numpy
import meteostat

print(numpy.__version__)
print(meteostat.__version__)

1.26.4
1.6.7


In [6]:
from meteostat import Hourly

Hourly.clear_cache()

print("Meteostat cache cleared")

Meteostat cache cleared


In [7]:
from meteostat import Hourly

print(Hourly.cache_dir)

C:\Users\DTMAR\.meteostat\cache


In [8]:
import shutil
from meteostat import Hourly

cache_dir = Hourly.cache_dir
print("Deleting:", cache_dir)

shutil.rmtree(cache_dir, ignore_errors=True)

print("Meteostat cache folder deleted")

Deleting: C:\Users\DTMAR\.meteostat\cache
Meteostat cache folder deleted


In [9]:
from meteostat import Point, Hourly
from datetime import datetime

auckland = Point(-36.8485, 174.7633)

start = datetime(2024, 1, 1)
end = datetime(2024, 1, 3)

weather = Hourly(auckland, start, end)
weather_df = weather.fetch()

print("Shape:", weather_df.shape)
display(weather_df.head())

Shape: (49, 11)


,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2024-01-01 00:00:00,19.0,10.3,57.0,0.0,NaN,217.0,29.9,NaN,1020.0,NaN,3.0
2024-01-01 01:00:00,19.1,10.1,56.0,0.0,NaN,220.0,28.4,NaN,1020.1,NaN,3.0
2024-01-01 02:00:00,19.0,10.0,56.0,0.0,NaN,217.0,25.9,NaN,1020.0,NaN,3.0
2024-01-01 03:00:00,18.9,9.9,56.0,0.0,NaN,215.0,24.5,NaN,1020.0,NaN,3.0
2024-01-01 04:00:00,18.7,9.5,55.0,0.0,NaN,210.0,24.1,NaN,1019.9,NaN,3.0


In [ ]:
#Inspect missing values
weather_df.isna().sum()

temp     0
dwpt     0
rhum     0
prcp     0
snow    49
wdir     0
wspd     0
wpgt    49
pres     0
tsun    49
coco     0
dtype: int64

In [13]:
#select useful feature only
weather_clean = weather_df[[
    "temp",   # temperature
    "rhum",   # humidity
    "prcp",   # precipitation (IMPORTANT)
    "wspd",   # wind speed
    "pres"    # pressure
]].copy()

In [ ]:
#Handle missing values
#forward fil
weather_clean = weather_clean.fillna(method="ffill")

In [17]:
#validate the cleaned data/print(weather_clean.shape)
weather_clean.isna().sum()
print("Shape:", weather_clean.shape)
weather_clean.head()



Shape: (49, 5)


,temp,rhum,prcp,wspd,pres
time,,,,,
2024-01-01 00:00:00,19.0,57.0,0.0,29.9,1020.0
2024-01-01 01:00:00,19.1,56.0,0.0,28.4,1020.1
2024-01-01 02:00:00,19.0,56.0,0.0,25.9,1020.0
2024-01-01 03:00:00,18.9,56.0,0.0,24.5,1020.0
2024-01-01 04:00:00,18.7,55.0,0.0,24.1,1019.9


In [18]:
#Keep datetime index
weather_clean.index

DatetimeIndex(['2024-01-01 00:00:00', '2024-01-01 01:00:00',
               '2024-01-01 02:00:00', '2024-01-01 03:00:00',
               '2024-01-01 04:00:00', '2024-01-01 05:00:00',
               '2024-01-01 06:00:00', '2024-01-01 07:00:00',
               '2024-01-01 08:00:00', '2024-01-01 09:00:00',
               '2024-01-01 10:00:00', '2024-01-01 11:00:00',
               '2024-01-01 12:00:00', '2024-01-01 13:00:00',
               '2024-01-01 14:00:00', '2024-01-01 15:00:00',
               '2024-01-01 16:00:00', '2024-01-01 17:00:00',
               '2024-01-01 18:00:00', '2024-01-01 19:00:00',
               '2024-01-01 20:00:00', '2024-01-01 21:00:00',
               '2024-01-01 22:00:00', '2024-01-01 23:00:00',
               '2024-01-02 00:00:00', '2024-01-02 01:00:00',
               '2024-01-02 02:00:00', '2024-01-02 03:00:00',
               '2024-01-02 04:00:00', '2024-01-02 05:00:00',
               '2024-01-02 06:00:00', '2024-01-02 07:00:00',
               '2024-01-

Goal is to align:

GTFS realtime timestamps (irregular)
Weather timestamps (hourly)

In [ ]:
#load realtime_with_routes_sample.csv

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

gtfs_path = PROJECT_ROOT / "data" / "interim" / "realtime_with_routes_sample.csv"

gtfs_df = pd.read_csv(gtfs_path)

print(gtfs_df.shape)
gtfs_df.head()

(1288, 14)


,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,route_short_name,route_long_name,route_type,transport_mode
0,521-93022-76500-1-23f81938,521-93022-76500-1-23f81938,MTID-241,1,21:15:00,20260430,1777414947,0,False,0.0,MTID,MTID,4.0,Ferry
1,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260430,1777482603,0,False,0.0,MTIA,MTIA,4.0,Ferry
2,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260430,1777482612,0,False,0.0,MTIA,MTIA,4.0,Ferry
3,248-840018-70020-2-6155400-1931d669,248-840018-70020-2-6155400-1931d669,ONE-201,1,19:27:00,20260430,1777535095,-12,False,-0.2,ONE,ONE,2.0,Train
4,248-840042-80820-2-6167400-d9ff341c,248-840042-80820-2-6167400-d9ff341c,ONE-201,1,22:27:00,20260430,1777484461,0,False,0.0,ONE,ONE,2.0,Train


In [20]:
#convert the timestamp
gtfs_df["timestamp_dt"] = pd.to_datetime(gtfs_df["timestamp"], unit="s")

gtfs_df["hour"] = gtfs_df["timestamp_dt"].dt.floor("H")

gtfs_df[["timestamp", "timestamp_dt", "hour", "delay_minutes"]].head()

,timestamp,timestamp_dt,hour,delay_minutes
0,1777414947,2026-04-28 22:22:27,2026-04-28 22:00:00,0.0
1,1777482603,2026-04-29 17:10:03,2026-04-29 17:00:00,0.0
2,1777482612,2026-04-29 17:10:12,2026-04-29 17:00:00,0.0
3,1777535095,2026-04-30 07:44:55,2026-04-30 07:00:00,-0.2
4,1777484461,2026-04-29 17:41:01,2026-04-29 17:00:00,0.0


In [21]:
#prepare weather data for merging
weather_ready = weather_clean.reset_index()
weather_ready = weather_ready.rename(columns={"time": "hour"})

weather_ready.head()

,hour,temp,rhum,prcp,wspd,pres
0,2024-01-01 00:00:00,19.0,57.0,0.0,29.9,1020.0
1,2024-01-01 01:00:00,19.1,56.0,0.0,28.4,1020.1
2,2024-01-01 02:00:00,19.0,56.0,0.0,25.9,1020.0
3,2024-01-01 03:00:00,18.9,56.0,0.0,24.5,1020.0
4,2024-01-01 04:00:00,18.7,55.0,0.0,24.1,1019.9


In [22]:
#merge 
merged_df = pd.merge(
    gtfs_df,
    weather_ready,
    on="hour",
    how="left"
)

print("Merged shape:", merged_df.shape)
merged_df[["hour", "delay_minutes", "temp", "prcp", "wspd"]].head()

Merged shape: (1288, 21)


,hour,delay_minutes,temp,prcp,wspd
0,2026-04-28 22:00:00,0.0,NaN,NaN,NaN
1,2026-04-29 17:00:00,0.0,NaN,NaN,NaN
2,2026-04-29 17:00:00,0.0,NaN,NaN,NaN
3,2026-04-30 07:00:00,-0.2,NaN,NaN,NaN
4,2026-04-29 17:00:00,0.0,NaN,NaN,NaN
